## Ch7-02 — State machine and execution traces

This notebook introduces state usage with transitions (construct 13); after running it you can simulate the toaster's operating cycle for a normal toast run and a cancelled run.


Chapter 4 introduced action flow for the `ApplyHeat` operation. This notebook adds a `state Cycle` that captures the toaster's discrete operating modes — idle, heating, ready, and cancelled — and uses `execute_state` to simulate how events move the system between those modes. See [Ch4-01 action def](../ch04-functional-decomp/01-action-def-ffbd.ipynb) for the action def this state machine complements.


In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch07-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch07-cumulative.sysml` file adds `state Cycle` with four substates (`idle`, `heating`, `ready`, `cancelled`) and three transitions (`idle → heating` on `Start`, `heating → ready` on `Finish`, `heating → cancelled` on `Cancel`). This is construct 13 — the first executable behavior in the model. `model.execute_state()` can trace event sequences through this state machine.

In [ ]:
# A state machine referencing an undefined transition target fails to parse.
bad_source = """
package P {
    item def Go;
    state S {
        entry; then a;
        state a;
        transition first a accept Go then missing_state;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for undefined transition target"
# Expected: diagnostic for 'missing_state' as an unresolved reference
print(f"Negative control ok: bad.ok={bad.ok}")


In [ ]:
# Locate the Cycle state machine by qualified name
cycle = model.find("ToasterDemo::Cycle")
assert cycle is not None, "Cycle not found"
print(f"Cycle: kind={cycle.kind!r}, id={cycle.id!r}")

# Normal run: Start → heating, Finish → ready
normal = model.execute_state(cycle.id, events=["Start", "Finish"])
print(f"Normal trace:  {normal['states_visited']}")
assert normal["states_visited"] == ["idle", "heating", "ready"]

# Cancelled run: Start → heating, Cancel → cancelled
cancelled = model.execute_state(cycle.id, events=["Start", "Cancel"])
print(f"Cancelled trace: {cancelled['states_visited']}")
assert cancelled["states_visited"] == ["idle", "heating", "cancelled"]

conn.close()


The `state Cycle` with four substates and three transitions (A-F) is executed by OpenSysML's `execute_state` (O-S); the states visited — `['idle', 'heating', 'ready']` for a normal run and `['idle', 'heating', 'cancelled']` for a cancel — appear in the result dict (E).


Try the chapter exercise in `exercises/ch07/exercise.ipynb`: add a `state BrewCycle` to the coffee maker model with an `Overheat` transition to a `fault` state, and verify the trace with `execute_state`.
